# Milestone 3 dan Final: EDA dan Insights

Notebook ini melanjutkan `02_data_cleaning.ipynb`. Inputnya data bersih hasil Milestone 2 di `data/processed/`, dan file itu tidak diubah di sini. Isinya analisis temporal dan spasial deteksi titik panas NASA FIRMS VIIRS S-NPP 375 m untuk Indonesia (2019-2024), dengan 7 visualisasi interaktif Plotly dan 6 insight utama.

Setiap grafik menjawab satu pertanyaan dan menyebut cakupan datanya. Angka di teks berasal dari output cell di notebook ini. Satu hal yang perlu diingat sepanjang notebook: yang dihitung adalah **deteksi satelit**, bukan jumlah kebakaran.

Plotly yang dipakai harus versi 6 atau lebih baru (membaca DataFrame Polars langsung). 

## 1. Setup dan Data

In [1]:
from datetime import date
from pathlib import Path

import duckdb
import plotly
import plotly.express as px
import plotly.graph_objects as go
import polars as pl


# cari folder repo (yang punya data/raw), karena kernel bisa jalan dari root atau dari notebooks/
def cari_root(mulai):
    for p in [mulai, *mulai.parents]:
        if (p / "data" / "raw").is_dir():
            return p
    raise FileNotFoundError("folder data/raw tidak ketemu")


PROJECT_ROOT = cari_root(Path.cwd().resolve())
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "firms_viirs_snpp_indonesia_bersih.parquet"
assert DATA_PATH.exists(), "jalankan 02_data_cleaning.ipynb dulu supaya file data bersih ada"

NAMA_BULAN = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)

print("Polars", pl.__version__, "| DuckDB", duckdb.__version__, "| Plotly", plotly.__version__)

Polars 1.44.2 | DuckDB 1.5.6 | Plotly 6.7.0


In [2]:
data = pl.scan_parquet(DATA_PATH)
total_baris = data.select(pl.len()).collect().item()

print(f"Baris: {total_baris:,} | kolom: {len(data.collect_schema())}")
pl.DataFrame({"kolom": data.collect_schema().names(), "tipe": [str(t) for t in data.collect_schema().dtypes()]})

Baris: 1,011,936 | kolom: 15


kolom,tipe
str,str
"""latitude""","""Float64"""
"""longitude""","""Float64"""
"""bright_ti4""","""Float64"""
"""scan""","""Float64"""
"""track""","""Float64"""
"""acq_date""","""Date"""
"""acq_time""","""String"""
"""confidence""","""String"""
"""bright_ti5""","""Float64"""


**Keputusan:** data dibaca lazy dengan `scan_parquet`. Semua agregasi dihitung di Polars dulu, baru hasil ringkasnya dikirim ke Plotly, jadi tidak ada satu pun grafik yang memplot 1.011.936 titik mentah.

**Hasil:** data bersih berisi 1.011.936 baris dan 15 kolom, termasuk `waktu_utc`, `waktu_wib`, dan `frp_tinggi` dari Milestone 2.

## 2. Persiapan Kolom Turunan

In [3]:
d = data.with_columns(
    pl.col("acq_date").dt.year().alias("tahun"),
    pl.col("acq_date").dt.month().alias("bulan"),
    pl.col("waktu_utc").dt.hour().alias("jam_utc"),
)

# uji: jam_utc harus sama dengan dua digit pertama acq_time (HHMM) di seluruh baris
beda_jam = d.filter(pl.col("jam_utc") != pl.col("acq_time").str.slice(0, 2).cast(pl.Int64)).select(pl.len()).collect().item()
print("baris dengan jam_utc tidak cocok acq_time:", beda_jam)
d.select("acq_date", "acq_time", "waktu_utc", "tahun", "bulan", "jam_utc").head(3).collect()

baris dengan jam_utc tidak cocok acq_time: 0


acq_date,acq_time,waktu_utc,tahun,bulan,jam_utc
date,str,datetime[μs],i32,i8,i8
2019-01-01,"""0410""",2019-01-01 04:10:00,2019,1,4
2019-01-01,"""0410""",2019-01-01 04:10:00,2019,1,4
2019-01-01,"""0410""",2019-01-01 04:10:00,2019,1,4


**Keputusan:** `tahun`, `bulan`, dan `jam_utc` dibuat di memori saja. Analisis jam memakai UTC, bukan `waktu_wib`, karena `waktu_wib` hanya UTC+7 dan menggeser jam untuk wilayah WITA dan WIT.

**Hasil:** tidak ada baris dengan `jam_utc` yang berbeda dari dua digit pertama `acq_time`.

## 3. Pola Temporal

Cakupan data pada semua grafik di seksi ini: seluruh baris, kecuali dinyatakan lain.

### Grafik 1. Bagaimana jumlah deteksi per bulan berubah dari 2019 sampai 2024?

In [4]:
semua_hari = pl.DataFrame(
    {"tanggal": pl.date_range(date(2019, 1, 1), date(2024, 12, 31), interval="1d", eager=True)}
).with_columns(
    pl.col("tanggal").dt.year().alias("tahun"),
    pl.col("tanggal").dt.month().alias("bulan"),
)
hari_kalender = semua_hari.group_by("tahun", "bulan").len().rename({"len": "hari_kalender"})

per_bulan = (
    d.group_by("tahun", "bulan")
    .agg(pl.len().alias("deteksi"), pl.col("acq_date").n_unique().alias("hari_ada_data"))
    .collect()
    .join(hari_kalender, on=["tahun", "bulan"])
    .with_columns(
        (pl.col("hari_kalender") - pl.col("hari_ada_data")).alias("hari_tanpa_data"),
        pl.date(pl.col("tahun"), pl.col("bulan"), 1).alias("bulan_awal"),
    )
    .sort("bulan_awal")
)

fig = px.line(
    per_bulan,
    x="bulan_awal",
    y="deteksi",
    markers=True,
    hover_data=["hari_tanpa_data"],
    labels={"bulan_awal": "Bulan", "deteksi": "Jumlah deteksi", "hari_tanpa_data": "Hari tanpa data"},
    title="Jumlah deteksi titik panas per bulan, 2019-2024",
)
fig.update_layout(xaxis_rangeslider_visible=True)
fig.show()

**Hasil:** puncak bulanan tiap tahun: 2019 September (197.018), 2020 Agustus (20.301), 2021 Oktober (13.921), 2022 September (10.935), 2023 Oktober (78.784), 2024 September (33.586). Puncak selalu jatuh pada Agustus sampai Oktober.

**Catatan:** tooltip menampilkan `hari_tanpa_data` per bulan. Bulan dengan hari tanpa data (mis. Agustus 2022 dan Juli 2024) bisa tampak lebih rendah dari yang sebenarnya, jadi perbandingan bulan-ke-bulan di bulan itu perlu hati-hati.

### Grafik 2. Apakah pola musiman sama di semua tahun?

In [5]:
matriks = per_bulan.pivot(on="tahun", index="bulan", values="deteksi").sort("bulan")
kolom_tahun = sorted(k for k in matriks.columns if k != "bulan")

fig = go.Figure(
    go.Heatmap(
        z=[list(r) for r in matriks.select(kolom_tahun).rows()],
        x=kolom_tahun,
        y=NAMA_BULAN,
        colorscale="YlOrRd",
        texttemplate="%{z:,}",
        hovertemplate="Tahun %{x}, %{y}: %{z:,} deteksi<extra></extra>",
        colorbar={"title": "Deteksi"},
    )
)
fig.update_layout(title="Deteksi titik panas per bulan dan tahun", xaxis_title="Tahun", yaxis={"autorange": "reversed"})
fig.show()

**Hasil:** Agustus sampai Oktober menyumbang 71,7% dari seluruh deteksi (September saja 34,0%). September 2019 adalah bulan-tahun dengan deteksi terbanyak: 197.018 deteksi, atau 19,5% dari seluruh data enam tahun.

**Catatan:** pola ini menunjukkan *kapan* deteksi terkonsentrasi, bukan *kenapa*. Penyebabnya (mis. kondisi iklim) tidak bisa disimpulkan dari data deteksi saja dan belum dianalisis di sini.

### Grafik 3. Seberapa besar deteksi `type` 0 dibanding `type` lain di tiap tahun?

In [6]:
nama_type = {
    0: "0 dugaan kebakaran vegetasi",
    1: "1 gunung api aktif",
    2: "2 sumber statis lain di darat",
    3: "3 lepas pantai",
}

per_tahun_type = (
    d.group_by("tahun", "type")
    .agg(pl.len().alias("deteksi"))
    .collect()
    .with_columns(pl.col("type").replace_strict(nama_type, return_dtype=pl.String).alias("jenis"))
    .sort("tahun", "type")
)

fig = px.bar(
    per_tahun_type,
    x="tahun",
    y="deteksi",
    color="jenis",
    barmode="group",
    log_y=True,
    category_orders={"jenis": list(nama_type.values())},
    labels={"tahun": "Tahun", "deteksi": "Jumlah deteksi (skala log)", "jenis": "Jenis (type)"},
    title="Deteksi per tahun menurut type (sumbu y skala logaritmik)",
)
fig.update_xaxes(type="category")
fig.show()

per_tahun_type.pivot(on="type", index="tahun", values="deteksi").sort("tahun")

tahun,0,1,2,3
i32,u32,u32,u32,u32
2019,424533,4252,7189,1307
2020,81518,2507,6860,651
2021,58784,2396,6661,484
2022,47333,2453,7040,394
2023,236123,4494,11812,959
2024,89940,2857,10785,604


**Hasil:** deteksi `type` 0 sangat berubah-ubah antar tahun, dari 47.333 (2022) sampai 424.533 (2019), sekitar 9,0 kali lipat. Deteksi `type` selain 0 jauh lebih stabil: 9.541 (2021) sampai 17.265 (2023), sekitar 1,8 kali lipat.

**Catatan:** definisi `type` mengikuti dokumentasi NASA (0 dugaan kebakaran vegetasi, 1 gunung api aktif, 2 sumber statis lain di darat, 3 deteksi lepas pantai yang mencakup semua deteksi di atas air). Sumber: https://modaps.modaps.eosdis.nasa.gov/services/about/products/viirs-land-c2-nrt/vnp14imgtdl_nrt.html. Sumbu y memakai skala log supaya kedua kelompok bisa terlihat sekaligus, jadi tinggi batang tidak sebanding secara linear. Grafik ini memakai semua baris dan hanya memisahkannya menurut `type`; grafik lain tidak menyaring `type`.

### Grafik 4. Pada jam berapa (UTC) deteksi tercatat?

In [7]:
nama_dn = {"D": "Siang (D)", "N": "Malam (N)"}

per_jam = (
    d.group_by("jam_utc", "daynight")
    .agg(pl.len().alias("deteksi"))
    .collect()
    .with_columns(pl.col("daynight").replace_strict(nama_dn, return_dtype=pl.String).alias("waktu"))
    .sort("jam_utc")
)

fig = px.bar(
    per_jam,
    x="jam_utc",
    y="deteksi",
    color="waktu",
    labels={"jam_utc": "Jam deteksi (UTC)", "deteksi": "Jumlah deteksi", "waktu": "Siang/malam"},
    title="Deteksi per jam UTC menurut siang (D) dan malam (N)",
)
fig.update_xaxes(dtick=1)
fig.show()

per_jam.pivot(on="daynight", index="jam_utc", values="deteksi").sort("jam_utc")

jam_utc,D,N
i8,u32,u32
3,16554,null
4,94632,null
5,306465,null
6,247752,null
7,31198,null
15,null,1966
16,null,28413
17,null,141043
18,null,127330


**Hasil:** deteksi hanya muncul pada 10 jam UTC berbeda, terpisah menjadi dua jendela: siang pada jam 03 sampai 07 UTC (puncak jam 05 dengan 306.465 dan jam 06 dengan 247.752) dan malam pada jam 15 sampai 19 UTC (puncak jam 17 dengan 141.043 dan jam 18 dengan 127.330). Deteksi siang 68,8% dan malam 31,2%.

**Catatan:** dua jendela sempit ini konsisten dengan deteksi yang hanya terjadi saat satelit lewat, sehingga pola jam kemungkinan besar mencerminkan jadwal lintasan satelit dan belum bisa dibaca sebagai kapan api menyala. Jadwal lintasannya belum diverifikasi ke dokumentasi, jadi tidak ada kesimpulan tentang pola harian api yang dinyatakan di sini.

## 4. Pola Spasial

### Grafik 5. Di mana deteksi terkonsentrasi dan bagaimana bergeser antar tahun?
Peta kerapatan tanpa peta dasar: tiap titik adalah kotak 0,5 x 0,5 derajat (titik mewakili sudut barat daya kotak), warnanya skala log dari jumlah deteksi, dan penggeser di bawah memilih tahun. Cakupan: seluruh baris.

In [8]:
sel_peta = (
    d.with_columns(
        ((pl.col("latitude") * 2).floor() / 2).alias("lintang"),
        ((pl.col("longitude") * 2).floor() / 2).alias("bujur"),
    )
    .group_by("tahun", "lintang", "bujur")
    .agg(pl.len().alias("deteksi"))
    .sort("tahun")
    .collect()
    .with_columns(pl.col("deteksi").log10().alias("log10_deteksi"))
)
print("kotak 0,5 derajat yang terisi (tahun x lokasi):", sel_peta.height)

fig = px.scatter(
    sel_peta,
    x="bujur",
    y="lintang",
    color="log10_deteksi",
    animation_frame="tahun",
    color_continuous_scale="YlOrRd",
    range_color=[0, sel_peta["log10_deteksi"].max()],
    hover_data={"deteksi": True, "log10_deteksi": False},
    labels={"bujur": "Bujur", "lintang": "Lintang", "log10_deteksi": "log10(deteksi)"},
    title="Kerapatan deteksi per kotak 0,5 derajat",
)
fig.update_traces(marker={"size": 7, "symbol": "square"})
for frame in fig.frames:
    frame.data[0].marker.size = 7
    frame.data[0].marker.symbol = "square"
fig.update_xaxes(range=[95, 141])
fig.update_yaxes(range=[-11, 6], scaleanchor="x", scaleratio=1)
fig.show()

kotak 0,5 derajat yang terisi (tahun x lokasi): 5355


In [9]:
# angka pendukung peta: kotak 1 derajat
sel_1derajat = (
    d.with_columns(pl.col("latitude").floor().alias("lintang"), pl.col("longitude").floor().alias("bujur"))
    .group_by("lintang", "bujur")
    .agg(pl.len().alias("deteksi"))
    .sort("deteksi", descending=True)
    .collect()
)
print("kotak 1 derajat terisi:", sel_1derajat.height)
for k in (10, 50):
    print(f"{k} kotak teratas = {sel_1derajat.head(k)['deteksi'].sum() / total_baris * 100:.1f}% dari semua deteksi")
print(sel_1derajat.head(3))

teratas_per_tahun = (
    d.with_columns(pl.col("latitude").floor().alias("lintang"), pl.col("longitude").floor().alias("bujur"))
    .group_by("tahun", "lintang", "bujur")
    .agg(pl.len().alias("deteksi"))
    .sort("deteksi", descending=True)
    .unique(subset="tahun", keep="first", maintain_order=True)
    .sort("tahun")
    .collect()
)
teratas_per_tahun

kotak 1 derajat terisi: 363
10 kotak teratas = 22.3% dari semua deteksi
50 kotak teratas = 60.7% dari semua deteksi
shape: (3, 3)
┌─────────┬───────┬─────────┐
│ lintang ┆ bujur ┆ deteksi │
│ ---     ┆ ---   ┆ ---     │
│ f64     ┆ f64   ┆ u32     │
╞═════════╪═══════╪═════════╡
│ -4.0    ┆ 105.0 ┆ 38297   │
│ -3.0    ┆ 114.0 ┆ 36533   │
│ -10.0   ┆ 124.0 ┆ 21138   │
└─────────┴───────┴─────────┘


tahun,lintang,bujur,deteksi
i32,f64,f64,u32
2019,-3.0,114.0,25687
2020,-10.0,124.0,3264
2021,-9.0,118.0,2937
2022,-3.0,121.0,1779
2023,-4.0,105.0,15626
2024,0.0,110.0,3310


**Hasil:** deteksi terkonsentrasi pada sedikit kotak. Dari 363 kotak 1 derajat yang terisi, 10 kotak teratas menyumbang 22,3% dan 50 kotak teratas 60,7% dari seluruh deteksi. Dua kotak teratas (sudut barat daya lintang, bujur) adalah (-4, 105) dengan 38.297 deteksi dan (-3, 114) dengan 36.533. Kotak teratas tiap tahun berpindah-pindah: 2019 (-3, 114), 2020 (-10, 124), 2021 (-9, 118), 2022 (-3, 121), 2023 (-4, 105), 2024 (0, 110).

**Catatan:** koordinat saja tidak menyebut nama wilayah, karena proyek ini belum memakai data batas wilayah, jadi penamaan daerah belum dilakukan. Kotak adalah pembagian koordinat sederhana dan bukan batas administratif.

## 5. Intensitas (FRP)

FRP pada resolusi 375 m hanya andal untuk api kecil atau berintensitas rendah menurut dokumentasi NASA (https://earthdata.nasa.gov/data/tools/firms/active-fire-data-attributes-modis-viirs), jadi nilai FRP ekstrem tidak otomatis berarti kebakaran besar. Karena itu analisis di sini memakai agregat dan persentase, bukan nilai tunggal.

### Grafik 6. Bagaimana sebaran FRP pada tiap kelas `confidence`?

In [10]:
batas_frp = [1, 2, 5, 10, 20, 50, 100, 200, 500]
label_frp = ["<=1", "1-2", "2-5", "5-10", "10-20", "20-50", "50-100", "100-200", "200-500", ">500"]

kelas_frp = (
    d.select("confidence", pl.col("frp").cut(batas_frp, labels=label_frp).cast(pl.String).alias("kelas_frp"))
    .group_by("confidence", "kelas_frp")
    .len()
    .collect()
    .with_columns((pl.col("len") / pl.col("len").sum().over("confidence") * 100).alias("persen"))
)

fig = px.bar(
    kelas_frp,
    x="kelas_frp",
    y="persen",
    color="confidence",
    barmode="group",
    category_orders={"kelas_frp": label_frp, "confidence": ["low", "nominal", "high"]},
    hover_data={"len": True},
    labels={"kelas_frp": "Kelas FRP (MW)", "persen": "Persen deteksi dalam kelas confidence", "len": "Jumlah"},
    title="Sebaran FRP menurut confidence",
)
fig.show()

d.group_by("confidence").agg(
    pl.len().alias("deteksi"),
    pl.col("frp").median().alias("median_frp"),
    pl.col("frp").quantile(0.9).alias("p90_frp"),
).sort("median_frp").collect()

confidence,deteksi,median_frp,p90_frp
str,u32,f64,f64
"""nominal""",909881,4.34,13.86
"""low""",69397,5.26,19.75
"""high""",32658,14.03,37.15


**Hasil:** median FRP tertinggi pada `confidence` high (14,03 MW), diikuti low (5,26 MW) dan nominal (4,34 MW). Persentase tiap kelas dihitung di dalam masing-masing `confidence`, jadi tinggi batang tidak dipengaruhi perbedaan jumlah deteksi antar kelas.

**Catatan:** menurut dokumentasi NASA, `confidence` high terkait piksel jenuh, low (siang) terkait sun glint dan anomali suhu yang lebih kecil, dan nominal bebas dari sun glint dengan anomali kuat. Karena definisinya berbeda-beda per kelas, hubungan ini deskriptif dan tidak menunjukkan sebab-akibat. Kelas dibatasi kanan (mis. "1-2" berarti lebih dari 1 sampai 2 MW).

### Grafik 7. Pada bulan apa deteksi ber-FRP tinggi paling sering muncul?

In [11]:
frp_tinggi_bulan = (
    d.group_by("bulan")
    .agg(pl.len().alias("deteksi"), pl.col("frp_tinggi").sum().alias("jumlah_frp_tinggi"))
    .sort("bulan")
    .collect()
    .with_columns((pl.col("jumlah_frp_tinggi") / pl.col("deteksi") * 100).round(2).alias("persen_frp_tinggi"))
)

fig = px.bar(
    frp_tinggi_bulan,
    x="bulan",
    y="persen_frp_tinggi",
    hover_data=["deteksi", "jumlah_frp_tinggi"],
    labels={"bulan": "Bulan", "persen_frp_tinggi": "Persen deteksi frp_tinggi", "jumlah_frp_tinggi": "Jumlah frp_tinggi"},
    title="Persentase deteksi frp_tinggi per bulan (semua tahun)",
)
fig.update_xaxes(tickmode="array", tickvals=list(range(1, 13)), ticktext=NAMA_BULAN)
fig.show()

frp_tinggi_bulan

bulan,deteksi,jumlah_frp_tinggi,persen_frp_tinggi
i8,u32,u32,f64
1,12358,85,0.69
2,22510,326,1.45
3,30653,502,1.64
4,22313,338,1.51
5,21272,174,0.82
6,26421,131,0.5
7,56725,806,1.42
8,160465,3726,2.32
9,344061,8401,2.44


**Hasil:** `frp_tinggi` (FRP di atas 39,88 MW dari IQR pada log1p, hasil Milestone 2) berjumlah 20.120 baris. Persentasenya tertinggi pada September (2,44%) dan terendah pada Juni (0,50%).

**Catatan:** `frp_tinggi` adalah penanda statistik dari Milestone 2, bukan label kebakaran besar. Jumlah per bulan mengikuti jumlah deteksi, jadi persentase dipakai agar bulan ramai tidak otomatis tampak lebih tinggi.

## 6. Cek Silang

Dua angka kunci dihitung ulang lewat DuckDB langsung dari file Parquet dan dibandingkan dengan hasil Polars.

In [12]:
con = duckdb.connect()
file_sql = DATA_PATH.as_posix()

duck_per_tahun = con.sql(
    f"SELECT year(acq_date) AS tahun, count(*) AS n FROM read_parquet('{file_sql}') GROUP BY 1 ORDER BY 1"
).fetchall()
polars_per_tahun = d.group_by("tahun").len().sort("tahun").collect().rows()

duck_tinggi = con.sql(f"SELECT count(*) FROM read_parquet('{file_sql}') WHERE frp_tinggi").fetchone()[0]
polars_tinggi = d.select(pl.col("frp_tinggi").sum()).collect().item()

print("deteksi per tahun, DuckDB :", duck_per_tahun)
print("deteksi per tahun, Polars :", polars_per_tahun)
print("sama:", duck_per_tahun == polars_per_tahun)
print(f"frp_tinggi, DuckDB: {duck_tinggi:,} | Polars: {polars_tinggi:,} | sama: {duck_tinggi == polars_tinggi}")
print("jumlah deteksi per bulan (Grafik 1) sama dengan total baris:", per_bulan["deteksi"].sum() == total_baris)

deteksi per tahun, DuckDB : [(2019, 437281), (2020, 91536), (2021, 68325), (2022, 57220), (2023, 253388), (2024, 104186)]
deteksi per tahun, Polars : [(2019, 437281), (2020, 91536), (2021, 68325), (2022, 57220), (2023, 253388), (2024, 104186)]
sama: True
frp_tinggi, DuckDB: 20,120 | Polars: 20,120 | sama: True
jumlah deteksi per bulan (Grafik 1) sama dengan total baris: True


**Hasil:** jumlah deteksi per tahun dan jumlah `frp_tinggi` dari DuckDB sama persis dengan Polars, dan total deteksi per bulan pada Grafik 1 sama dengan total baris.

## 7. Pemeriksaan Rubrik Milestone 3

Template Milestone 3 meminta: minimal 6 visualisasi interaktif, minimal satu operasi temporal/window, dan minimal 5 insight. Tiga butir yang belum ada di seksi sebelumnya ditambahkan di sini.

| Syarat template | Ada di notebook ini |
|---|---|
| Line chart tren waktu + range slider | Grafik 1 (slider ditambahkan) |
| Distribusi variabel utama (box/histogram/violin) | Grafik 8 |
| Korelasi antar-variabel numerik (heatmap) | Grafik 9 |
| Komposisi kategori | Grafik 3 dan Grafik 6 |
| Siklus musiman | Grafik 2 dan Grafik 7 |
| Pola spasial | Grafik 5 |
| Operasi temporal / window | `over("confidence")` di Seksi 6, `dt.year/month/hour` di Seksi 2 |
| 5 insight | 7 insight di Seksi 8 |


In [13]:
# Grafik 8: distribusi FRP per confidence.
# Box plot digambar dari kuantil hasil agregasi, bukan dari 1 juta baris mentah.
stat_frp = (
    d.group_by("confidence")
    .agg(
        pl.col("frp").quantile(0.01).alias("p1"),
        pl.col("frp").quantile(0.25).alias("q1"),
        pl.col("frp").median().alias("median"),
        pl.col("frp").quantile(0.75).alias("q3"),
        pl.col("frp").quantile(0.99).alias("p99"),
        pl.len().alias("deteksi"),
    )
    .collect()
    .sort("median")
)

fig = go.Figure()
for r in stat_frp.iter_rows(named=True):
    fig.add_trace(
        go.Box(
            name=r["confidence"],
            q1=[r["q1"]],
            median=[r["median"]],
            q3=[r["q3"]],
            lowerfence=[r["p1"]],
            upperfence=[r["p99"]],
            customdata=[[r["deteksi"]]],
            hovertemplate="%{x}<br>q1 %{q1:.2f} | median %{median:.2f} | q3 %{q3:.2f}<br>jumlah %{customdata[0]:,}<extra></extra>",
        )
    )
fig.update_layout(
    title=f"Distribusi FRP menurut confidence (kuartil seluruh {total_baris:,} baris)",
    yaxis={"type": "log", "title": "FRP (MW, skala log)"},
    xaxis_title="Confidence",
    showlegend=False,
)
fig.show()

stat_frp


confidence,p1,q1,median,q3,p99,deteksi
str,f64,f64,f64,f64,f64,u32
"""nominal""",0.46,2.25,4.34,7.67,49.03,909881
"""low""",1.22,3.36,5.26,9.07,105.03,69397
"""high""",4.73,9.66,14.03,21.84,140.72,32658


In [14]:
# Grafik 9: korelasi Pearson antar-kolom numerik, dihitung DuckDB langsung dari Parquet.
kolom_korelasi = {
    "frp": "frp",
    "bright_ti4": "bright_ti4",
    "bright_ti5": "bright_ti5",
    "scan": "scan",
    "track": "track",
    "jam_utc": "hour(waktu_utc)",
}
nama = list(kolom_korelasi)
pasangan = [(a, b) for i, a in enumerate(nama) for b in nama[i + 1 :]]
ekspresi = ", ".join(
    f'corr({kolom_korelasi[a]}, {kolom_korelasi[b]}) AS "{a}|{b}"' for a, b in pasangan
)
nilai = con.sql(f"SELECT {ekspresi} FROM read_parquet('{file_sql}')").fetchone()

# isi[i][j] = korelasi kolom i dengan kolom j; matriks simetris, diagonal 1
isi = {a: {b: 1.0 if a == b else None for b in nama} for a in nama}
for (a, b), v in zip(pasangan, nilai):
    isi[a][b] = v
    isi[b][a] = v

matriks_korelasi = pl.DataFrame({a: [isi[a][b] for b in nama] for a in nama})

fig = px.imshow(
    matriks_korelasi.to_numpy(),
    x=nama,
    y=nama,
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    text_auto=".2f",
    title="Korelasi Pearson antar-variabel numerik",
)
fig.update_layout(coloraxis_colorbar={"title": "r"})
fig.show()

matriks_korelasi


frp,bright_ti4,bright_ti5,scan,track,jam_utc
f64,f64,f64,f64,f64,f64
1.0,0.333891,0.223587,0.086251,0.031815,-0.211191
0.333891,1.0,0.518802,-0.051206,0.001648,-0.728115
0.223587,0.518802,1.0,-0.154853,-0.318695,-0.532364
0.086251,-0.051206,-0.154853,1.0,0.462785,0.026801
0.031815,0.001648,-0.318695,0.462785,1.0,0.038159
-0.211191,-0.728115,-0.532364,0.026801,0.038159,1.0


**Hasil:** median FRP naik konsisten mengikuti `confidence`: nominal 4,34 MW (909.881 baris), low 5,26 MW (69.397), high 14,03 MW (32.658). Korelasi terkuat adalah `bright_ti4` dengan `jam_utc` (r = -0,73) dan `bright_ti5` dengan `jam_utc` (r = -0,53); `frp` hanya berkorelasi sedang dengan `bright_ti4` (r = 0,33) dan lemah dengan `scan` (0,09) serta `track` (0,03).

Dua catatan keterbatasan:
- `jam_utc` adalah variabel melingkar (0-23), sehingga korelasi linear terhadap jam hanya indikasi kasar, bukan ukuran yang tepat.
- Arti fisis `bright_ti4`/`bright_ti5` dan `scan`/`track` belum diverifikasi dari dokumentasi resmi NASA; angka korelasi di atas hanya menunjukkan pola di data ini.


## 8. Insight Utama


In [15]:
# angka pendukung untuk insight di bawah, semuanya dari data bersih
per_tahun = d.group_by("tahun").len().sort("tahun").collect().with_columns((pl.col("len") / total_baris * 100).round(1).alias("persen"))
print(per_tahun)

gabungan = per_tahun.filter(pl.col("tahun").is_in([2019, 2023]))["len"].sum() / total_baris * 100
rasio_2019_2022 = per_tahun.filter(pl.col("tahun") == 2019)["len"].item() / per_tahun.filter(pl.col("tahun") == 2022)["len"].item()
sep_2019 = per_bulan.filter((pl.col("tahun") == 2019) & (pl.col("bulan") == 9))["deteksi"].item()
print(f"2019 + 2023 = {gabungan:.1f}% | 2019 dibanding 2022 = {rasio_2019_2022:.1f} kali")
print(f"September 2019 = {sep_2019:,} deteksi = {sep_2019 / total_baris * 100:.1f}% dari semua deteksi")

type_0 = per_tahun_type.filter(pl.col("type") == 0)["deteksi"]
selain_0 = per_tahun_type.filter(pl.col("type") != 0).group_by("tahun").agg(pl.col("deteksi").sum()).sort("tahun")["deteksi"]
print(f"type 0 per tahun: {type_0.min():,} sampai {type_0.max():,} ({type_0.max() / type_0.min():.1f} kali)")
print(f"type selain 0 per tahun: {selain_0.min():,} sampai {selain_0.max():,} ({selain_0.max() / selain_0.min():.1f} kali)")

per_bulan_total = d.group_by("bulan").len().sort("bulan").collect()
agu_okt = per_bulan_total.filter(pl.col("bulan").is_in([8, 9, 10]))["len"].sum()
print(f"Agustus-Oktober: {agu_okt:,} = {agu_okt / total_baris * 100:.1f}% | September saja: {per_bulan_total.filter(pl.col('bulan') == 9)['len'].item() / total_baris * 100:.1f}%")

print("total hari tanpa data:", per_bulan["hari_tanpa_data"].sum())
print("hari tanpa data per tahun:")
print(per_bulan.group_by("tahun").agg(pl.col("hari_tanpa_data").sum()).sort("tahun"))
print("bulan dengan hari tanpa data terbanyak:")
print(per_bulan.sort("hari_tanpa_data", descending=True).select("tahun", "bulan", "hari_tanpa_data", "deteksi").head(4))

siang_malam = (
    d.group_by("daynight")
    .agg(pl.len().alias("deteksi"), pl.col("frp").median().alias("median_frp"))
    .with_columns((pl.col("deteksi") / total_baris * 100).round(1).alias("persen"))
    .collect()
)
print(siang_malam)

porsi_selain_type0 = per_tahun_type.group_by("tahun").agg(
    (pl.col("deteksi").filter(pl.col("type") != 0).sum() / pl.col("deteksi").sum() * 100).round(1).alias("persen_selain_type0")
).sort("tahun")
print(porsi_selain_type0)

frp_total = d.select(pl.col("frp").sum()).collect().item()
p99_frp = d.select(pl.col("frp").quantile(0.99)).collect().item()
frp_p99 = d.filter(pl.col("frp") >= p99_frp).select(pl.col("frp").sum()).collect().item()
print(f"1% deteksi dengan FRP tertinggi (>= {p99_frp:.2f} MW) menyumbang {frp_p99 / frp_total * 100:.1f}% dari total FRP")

shape: (6, 3)
┌───────┬────────┬────────┐
│ tahun ┆ len    ┆ persen │
│ ---   ┆ ---    ┆ ---    │
│ i32   ┆ u32    ┆ f64    │
╞═══════╪════════╪════════╡
│ 2019  ┆ 437281 ┆ 43.2   │
│ 2020  ┆ 91536  ┆ 9.0    │
│ 2021  ┆ 68325  ┆ 6.8    │
│ 2022  ┆ 57220  ┆ 5.7    │
│ 2023  ┆ 253388 ┆ 25.0   │
│ 2024  ┆ 104186 ┆ 10.3   │
└───────┴────────┴────────┘
2019 + 2023 = 68.3% | 2019 dibanding 2022 = 7.6 kali
September 2019 = 197,018 deteksi = 19.5% dari semua deteksi
type 0 per tahun: 47,333 sampai 424,533 (9.0 kali)
type selain 0 per tahun: 9,541 sampai 17,265 (1.8 kali)
Agustus-Oktober: 725,101 = 71.7% | September saja: 34.0%
total hari tanpa data: 37
hari tanpa data per tahun:
shape: (6, 2)
┌───────┬─────────────────┐
│ tahun ┆ hari_tanpa_data │
│ ---   ┆ ---             │
│ i32   ┆ u32             │
╞═══════╪═════════════════╡
│ 2019  ┆ 0               │
│ 2020  ┆ 0               │
│ 2021  ┆ 0               │
│ 2022  ┆ 15              │
│ 2023  ┆ 2               │
│ 2024  ┆ 20              

**Insight 1. Deteksi sangat musiman: Agustus-Oktober menyumbang lebih dari dua pertiga seluruh deteksi.**
- *Angka:* Agustus sampai Oktober = 71,7% dari 1.011.936 deteksi, September saja 34,0%. Puncak bulanan tiap tahun selalu jatuh di Agustus sampai Oktober.
- *Grafik:* Grafik 1 dan 2.
- *Batas:* menunjukkan kapan deteksi terkonsentrasi, bukan penyebabnya. Beberapa bulan di musim itu punya hari tanpa data.

**Insight 2. Dua tahun mendominasi data: 2019 dan 2023.**
- *Angka:* 2019 = 437.281 deteksi (43,2%) dan 2023 = 253.388 (25,0%), jadi dua tahun itu 68,3% dari seluruh baris. September 2019 sendiri 197.018 deteksi (19,5% dari seluruh data). 2019 sekitar 7,6 kali lipat 2022.
- *Grafik:* Grafik 1, 2, dan 3.
- *Batas:* analisis gabungan enam tahun terutama mencerminkan 2019 dan 2023. Kesimpulan "rata-rata" sebaiknya dilihat per tahun juga.

**Insight 3. Deteksi `type` 0 jauh lebih berubah-ubah antar tahun dibanding `type` lain.**
- *Angka:* `type` 0 berkisar 47.333 sampai 424.533 per tahun (sekitar 9,0 kali lipat), sedangkan `type` selain 0 berkisar 9.541 sampai 17.265 (sekitar 1,8 kali lipat).
- *Grafik:* Grafik 3.
- *Angka tambahan:* porsi `type` selain 0 terhadap deteksi tahunan berkisar 2,9% (2019) sampai 17,3% (2022).
- *Batas:* pada tahun sepi deteksi `type` 0, porsi `type` selain 0 menjadi lebih besar. Analisis kebakaran vegetasi sebaiknya menyebut apakah memakai `type` 0 saja atau semua baris.

**Insight 4. Deteksi terkonsentrasi di sedikit lokasi dan lokasi puncaknya berpindah antar tahun.**
- *Angka:* dari 363 kotak 1 derajat terisi, 10 kotak teratas menyumbang 22,3% dan 50 kotak teratas 60,7% dari seluruh deteksi. Kotak teratas tiap tahun berbeda: 2019 (-3, 114), 2020 (-10, 124), 2021 (-9, 118), 2022 (-3, 121), 2023 (-4, 105), 2024 (0, 110).
- *Grafik:* Grafik 5.
- *Batas:* belum ada pemetaan ke nama wilayah. Kotak adalah pembagian koordinat sederhana.

**Insight 5. Jam deteksi hanya muncul di dua jendela sempit, yang konsisten dengan lintasan satelit dan bukan pola harian api.**
- *Angka:* siang pada jam 03-07 UTC (68,8%) dan malam pada jam 15-19 UTC (31,2%). Median FRP siang 5,99 MW dan malam 1,74 MW.
- *Grafik:* Grafik 4.
- *Batas:* jadwal lintasan satelit belum diverifikasi ke dokumentasi. Perbedaan FRP siang-malam bersifat deskriptif dan bukan bukti api lebih kuat di siang hari.

**Insight 6. Intensitas (FRP) sangat miring: sebagian kecil deteksi menyumbang porsi besar total FRP.**
- *Angka:* 1% deteksi dengan FRP tertinggi menyumbang 13,0% dari total FRP. Persentase `frp_tinggi` tertinggi pada September (2,44%) dan terendah pada Juni (0,50%). Median FRP menurut `confidence`: high 14,03, low 5,26, nominal 4,34 MW.
- *Grafik:* Grafik 6 dan 7.
- *Batas:* FRP resolusi 375 m hanya andal untuk api kecil atau berintensitas rendah (dokumentasi NASA), jadi nilai ekstrem tidak otomatis berarti kebakaran besar.

## Catatan Milestone 3

Rangkuman catatan metodologis dan batasan interpretasi analisis Milestone 3:

- **Keterbatasan kelengkapan waktu:** terdapat 37 hari kalender tanpa rekaman deteksi titik panas (2022 = 15 hari, 2023 = 2 hari, 2024 = 20 hari). Bulan dengan hari tanpa data terbanyak adalah Juli 2024 (11 hari) dan Agustus 2022 (10 hari). Oleh karena itu, perbandingan antar-bulan atau antar-tahun yang melibatkan periode tersebut perlu dilakukan dengan cermat.
- **Deteksi bukan kejadian kebakaran tunggal:** satu titik api di lapangan dapat terdeteksi berulang kali pada lintasan satelit berbeda, dan satu kejadian kebakaran besar dapat mencakup puluhan hingga ratusan piksel deteksi. Seluruh angka pada notebook ini mengacu pada unit **jumlah deteksi piksel sensorik**, bukan jumlah kejadian fisik kebakaran.
- **Pemisahan korelasi dan kausalitas:** analisis ini mendeskripsikan pola temporal dan spasial (*kapan* dan *di mana* deteksi terkonsentrasi), bukan membuktikan penyebabnya secara langsung. Faktor pendorong (seperti anomali iklim El Niño atau aktivitas pembukaan lahan) dicatat sebagai konteks domain yang membutuhkan data eksternal pendukung.
- **Cakupan data:** seluruh visualisasi dan insight utama memanfaatkan 1.011.936 baris data bersih secara lengkap tanpa membuang kategori `type`. Grafik 3 dan Insight 3 secara khusus membedah dinamika `type 0` (vegetasi) dibandingkan sumber panas lainnya.
- **Batasan spasial dan orbital:** pengelompokan spasial dilakukan berbasis grid koordinat matematis (0,5° dan 1,0°), bukan batas administratif daerah resmi karena keterbatasan ketiadaan shapefile wilayah terintegrasi. Selain itu, jendela waktu deteksi sangat terikat pada jadwal lintasan polar satelit Suomi NPP.